In [1]:
! pip install torch transformers

In [2]:
import numpy as np
import pandas as pd

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig


In [38]:
df=pd.read_csv('/kaggle/input/datasets/adithip2000/genre-generalised-and-tagged/final_processed_data.csv')

In [39]:
df.head()

,Unnamed: 0,genres,form,summary
0,0,"['sci-fi', 'drama', 'family']",book,"old major, the old boar on the manor farm, cal..."
1,1,"['sci-fi', 'drama']",book,"alex, a teenager living in near-future england..."
2,2,['drama'],book,the text of the plague is divided into five pa...
3,3,"['sci-fi', 'fantasy', 'drama']",book,the novel posits that space around the milky w...
4,4,"['sci-fi', 'fantasy', 'drama', 'family']",book,"ged is a young boy on gont, one of the larger ..."


In [40]:
test_data=pd.read_csv('/kaggle/input/datasets/adithip2000/test-data/test_cmu_data.csv')
test_data.head()

,Unnamed: 0,summary,genres,form
0,0,"A heart-wrenching tale of Choma, an untouchabl...",drama,book
1,1,A poignant cinematic portrayal of an untouchab...,drama,movie
2,2,"Set in the pre-independence Malnad region, the...",drama,book
3,2,"Set in the pre-independence Malnad region, the...",history,book
4,3,A visually rich adaptation capturing the fadin...,drama,movie


In [41]:
test_data['genres'].unique()

array(['drama', 'history', 'action', 'romance', 'adventure', 'thriller',
       'crime', 'family', 'comedy'], dtype=object)

In [7]:
# genre_map = {
#     # ---------------- DRAMA ----------------
#     "fiction": "drama",
#     "novel": "drama",
#     "drama": "drama",
#     "world cinema": "drama",

#     # ---------------- COMEDY ----------------
#     "comedy": "comedy",
#     "romantic comedy": "comedy",

#     # ---------------- ROMANCE ----------------
#     "romance novel": "romance",
#     "romance": "romance",
#     "romance film": "romance",
#     "romantic drama": "romance",

#     # ---------------- ACTION ----------------
#     "action": "action",
#     "action/adventure": "action",

#     # ---------------- ADVENTURE ----------------
#     "adventure": "adventure",
#     "adventure novel": "adventure",

#     # ---------------- THRILLER ----------------
#     "thriller": "thriller",
#     "suspense": "thriller",

#     # ---------------- HORROR ----------------
#     "horror": "horror",

#     # ---------------- SCI-FI ----------------
#     "science fiction": "sci-fi",
#     "speculative fiction": "sci-fi",
#     "dystopia": "sci-fi",
#     "alternate history": "sci-fi",

#     # ---------------- FANTASY ----------------
#     "fantasy": "fantasy",

#     # ---------------- CRIME ----------------
#     "crime fiction": "crime",

#     # ---------------- MYSTERY ----------------
#     "mystery": "mystery",
#     "detective fiction": "mystery",

#     # ---------------- HISTORY ----------------
#     "historical fiction": "history",
#     "historical novel": "history",

#     # ---------------- FAMILY ----------------
#     "children's literature": "family",
#     "family film": "family",
# }

In [42]:
df.shape

(49957, 4)

In [43]:
df.columns

Index(['Unnamed: 0', 'genres', 'form', 'summary'], dtype='object')

In [44]:
test_data.columns

Index(['Unnamed: 0', 'summary', 'genres', 'form'], dtype='object')

In [45]:
df.drop(['Unnamed: 0'],inplace=True,axis=1)
test_data.drop(['Unnamed: 0'],inplace=True,axis=1)

In [46]:
# from transformers import BertTokenizer as BToken
# tokenizer=BToken.from_pretrained('bert-base-uncased')
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
#model = AutoModel.from_pretrained(model_id)
# Move the model to GPU if available
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)


In [47]:
type(df['genres'].iloc[0])

str

In [48]:
df['genres'].iloc[0]

"['sci-fi', 'drama', 'family']"

In [49]:
test_data['genres'].iloc[61]

'drama'

In [50]:
import ast
df['genres']=df['genres'].apply(ast.literal_eval)

In [51]:
test_data['genres']=test_data['genres'].apply(lambda x: [x])

In [52]:
type(test_data['genres'].iloc[0])

list

In [53]:
test_data['genres'].iloc[0]

['drama']

In [54]:
label_names=sorted(df['genres'].explode().unique())

In [55]:
print(label_names)

['action', 'adventure', 'comedy', 'crime', 'drama', 'family', 'fantasy', 'history', 'horror', 'mystery', 'romance', 'sci-fi', 'thriller']


In [56]:
print(sorted(test_data['genres'].explode().unique()))

['action', 'adventure', 'comedy', 'crime', 'drama', 'family', 'history', 'romance', 'thriller']


In [57]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb=MultiLabelBinarizer()
y=mlb.fit_transform(df['genres'])
print(y[0])


[0 0 0 0 1 1 0 0 0 0 0 1 0]


In [58]:
y.shape

(49957, 13)

In [59]:
y[12]

array([0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

In [60]:
y_test_data=mlb.transform(test_data['genres'])

In [61]:
y_test_data[1]

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])

In [64]:
from torch.utils.data import Dataset
class GenreDataset(Dataset):
    def __init__(self, texts, labels,form,tokenizer, vectorizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.form=form
        self.tokenizer = tokenizer
        self.vectorizer = vectorizer
        self.max_len = max_len

        # 🔹 Use transform (NOT fit)
        bow_matrix = self.vectorizer.transform(self.texts)
        self.bow = torch.tensor(bow_matrix.toarray(), dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels[idx]
        form=self.form.iloc[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            "text":text,
            "form":form,
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "bow": self.bow[idx],
            "labels": torch.tensor(label, dtype=torch.float)
        }

In [65]:
label_counts=np.sum(y,axis=0)
for genre, count in zip(mlb.classes_, label_counts):
    print(genre, count)

action 6777
adventure 3599
comedy 10872
crime 5062
drama 27235
family 7090
fantasy 4426
history 1226
horror 4604
mystery 3614
romance 8919
sci-fi 7534
thriller 7925


In [66]:
total_samples=y.shape[0]
print(total_samples)


49957


In [67]:
X=df['summary']
# X=df['text']

In [69]:
X_test_data=test_data['summary']

In [70]:
from sklearn.model_selection import train_test_split as tts
X_train,X_test,Y_train,Y_test=tts(X,y,test_size=0.2,random_state=42)
X_val, X_test, Y_val, Y_test = tts(
    X_test,
    Y_test,
    test_size=0.5,
    random_state=42
)

In [71]:
print(X_train)

16256    in 2018 a prisoner named dante and his inmate ...
4204     the callum children spend their easter holiday...
6957     the enterprise is engaged in an unprecedented ...
9163     ike jerome, a 24-year-old architecture student...
14522    during a walk to the park raghuraman meets sud...
                               ...                        
11284    the novel continues on from the end of don't c...
44732    the film centers on themistocles and artemisia...
38158    cody weever grew up a child who loved films an...
860      the book's four main characters are ecological...
15795    the film begins with gayle, becky and judi per...
Name: summary, Length: 39965, dtype: object


In [72]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=8000,
    stop_words='english',
    min_df=5
)

vectorizer.fit(X_train)   # ONLY train

CountVectorizer(max_features=8000, min_df=5, stop_words='english')

In [ ]:
# train_dataset = GenreDataset(X_train, Y_train, tokenizer, vectorizer)
# val_dataset   = GenreDataset(X_val, Y_val, tokenizer, vectorizer)
# test_dataset  = GenreDataset(X_test, Y_test, tokenizer, vectorizer)

In [75]:
test_data_=GenreDataset(X_test_data,y_test_data,test_data['form'], tokenizer, vectorizer)

In [76]:
from torch.utils.data import DataLoader

In [78]:
vocab_size = len(vectorizer.get_feature_names_out())
bow_dim=vocab_size
vocab=vectorizer.get_feature_names_out()

In [92]:
test_loader = DataLoader(
    test_data_,
    batch_size=16,
    shuffle=False
)

In [93]:
len(test_loader)

4

In [80]:
import torch
import torch.nn as nn
from transformers import AutoModel

class topicGenreModel(nn.Module):
    def __init__(self, num_topics, num_genres, bow_dim, vocab_size, model_name="answerdotai/ModernBERT-base"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        bert_dim = self.backbone.config.hidden_size
        self.bow_weights = nn.Parameter(torch.tensor(0.1))
        
        self.context_encoder = nn.Sequential(
            nn.Linear(bert_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.bow_encoder = nn.Sequential(
            nn.Linear(vocab_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 🔹 Latent space (topics)
        self.fc_mu = nn.Linear(512, num_topics)
        self.fc_logvar = nn.Linear(512, num_topics)

        # 🔹 Topic → word matrix
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size))

        # 🔹 Topic → genre
        self.classifier = nn.Sequential(
            nn.Linear(num_topics + 256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_genres)
        )

    # ---------- Encode ----------
    def encode(self, bow, embedding):
        h_ctx = torch.relu(self.context_encoder(embedding))
        alpha = torch.sigmoid(self.bow_weights)
        
        if bow is not None:
            h_bow = torch.relu(self.bow_encoder(bow))
            if self.training:
                # Dropout-like mask for BoW features
                mask = (torch.rand(h_bow.size(0), 1).to(h_bow.device) > 0.3).float()
                h_bow = h_bow * mask
            h = torch.cat([h_ctx, alpha * h_bow], dim=1)
        else:
            h_bow = torch.zeros_like(h_ctx)
            h = torch.cat([h_ctx, h_bow], dim=1)
            
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar, h_ctx

    # ---------- Reparameterization ----------
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------- Decode ----------
    def decode(self, topic_dist):
        logits = torch.matmul(topic_dist, self.beta)
        return torch.softmax(logits, dim=1)

    # ---------- Forward ----------
    def forward(self, input_ids, attention_mask, bow):
        # 🔹 BERT embedding (using CLS token)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        embedding = outputs.last_hidden_state[:, 0, :]

        # 🔹 Encode
        mu, logvar, h_ctx = self.encode(bow, embedding)

        # 🔹 Sample topics
        z = self.reparameterize(mu, logvar)
        topic_dist = torch.softmax(z, dim=1)

        # 🔹 Genre prediction
        # Detaching topic distribution to prevent it from dominating the backbone gradients
        topic_dist_detached = topic_dist.detach()
        combined = torch.cat([h_ctx, 0.5 * topic_dist_detached], dim=1)
        genre_logits = self.classifier(combined)

        # 🔹 Word reconstruction
        word_dist = self.decode(topic_dist)

        return topic_dist, genre_logits, word_dist, mu, logvar

In [81]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

   return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "bow": self.bow[idx],
            "labels": torch.tensor(label, dtype=torch.float)
        }

Remember, whole batch will be processed at once from the loader

In [83]:
# vocab_size = len(vectorizer.get_feature_names_out())
# bow_dim=train_dataset.bow.shape[1]
num_genres=len(mlb.classes_)
print(f"Vocab size {vocab_size}")
print(f"bow dim {bow_dim}")
print(f"num_genres {num_genres}")
num_topics=32
print(f"num topics {num_topics}")
model=topicGenreModel(num_topics,num_genres,bow_dim,vocab_size)
state_dict=torch.load('/kaggle/input/models/adithip2000/final-ver/pytorch/default/1/checkpoint_v8_on_v7.pt',map_location=device)
model.load_state_dict(state_dict)
model.to(device)

    
    
    

Vocab size 8000
bow dim 8000
num_genres 13
num topics 32


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


topicGenreModel(
  (backbone): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (attn_norm): Lay

In [96]:
def inspect_sample(model, dataloader, vocab, label_names, device, threshold=0.2, idx=0, original_texts=None):
    
    model.eval()
    
    # Get one batch
    batch = next(iter(dataloader))
    
    # Select sample
    input_ids = batch['input_ids'][idx].unsqueeze(0).to(device)
    attention_mask = batch['attention_mask'][idx].unsqueeze(0).to(device)
    bow = batch['bow'][idx].unsqueeze(0).to(device)
    labels = batch['labels'][idx].to(device)
    text=batch['text'][idx]
    form=batch['form'][idx]
    
    with torch.no_grad():
        topic_dist, genre_logits, word_dist, mu, logvar = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bow=bow
        )
    
    # Predictions
    probs = torch.sigmoid(genre_logits)
    preds = (probs > threshold).int()
    
    # Convert to label names
    predicted_labels = [label_names[i] for i in range(len(label_names)) if preds[0][i] == 1]
    true_labels = [label_names[i] for i in range(len(label_names)) if labels[i] == 1]
    
    # Topic info
    topic_distribution = topic_dist[0].cpu().numpy()
    top_topic = torch.argmax(topic_dist, dim=1).item()
    
    # Top words
    def get_top_words(word_dist, vocab, top_k=10):
        top_indices = word_dist.squeeze().argsort(descending=True)[:top_k]
        return [vocab[i] for i in top_indices]
    
    top_words = get_top_words(word_dist, vocab)
    
    # Return values (useful if needed)
    return {
        "text":text,
        "form":form,
        "true_labels": true_labels,
        "predicted_labels": predicted_labels,
        "topic_distribution": topic_distribution,
        "top_topic": top_topic,
        "top_words": top_words
    }

In [94]:
for batch in test_loader:
    for i in range(0,len(batch)):
        print("=================================================")
        print(inspect_sample(
    model=model,
    dataloader=test_loader,
    vocab=vocab,
    label_names=label_names,
    device=device,
    threshold=0.5,
    idx=i,
      # optional
))

{'text': 'A heart-wrenching tale of Choma, an untouchable bonded laborer whose only solace is playing his drum, as he struggles against the oppressive caste system in a rural village.', 'form': 'book', 'true_labels': ['drama'], 'predicted_labels': ['drama'], 'topic_distribution': array([0.00588725, 0.00716426, 0.01942398, 0.01781542, 0.00464861,
       0.04672855, 0.2872587 , 0.01033369, 0.00346397, 0.02232333,
       0.054191  , 0.01302769, 0.01937988, 0.05701146, 0.00412389,
       0.00716174, 0.08575521, 0.01469398, 0.00719179, 0.01097707,
       0.02138145, 0.01775213, 0.11316463, 0.02590611, 0.0026854 ,
       0.03538746, 0.00631462, 0.03547547, 0.01496761, 0.00652545,
       0.01355036, 0.00832783], dtype=float32), 'top_topic': 6, 'top_words': ['men', 'death', 'world', 'son', 'attempt', 'ex', 'present', 'eventually', 'friends', 'war']}
{'text': "A poignant cinematic portrayal of an untouchable bonded laborer's tragic life, highlighting his struggles with caste prejudice, poverty,